In [ ]:
# STEP 1: Import libraries and define file paths
# Description: Set up environment and key file paths

import os
import pandas as pd

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
efficiency_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/heating_efficiency.csv"
models_base_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [20]:
# STEP 2: Load datasets
# Description: Read EPC dataset and heating efficiency lookup table

epc_df = pd.read_csv(epc_path)
eff_df = pd.read_csv(efficiency_path)

# Clean column names just in case
epc_df.columns = epc_df.columns.str.strip()
eff_df.columns = eff_df.columns.str.strip()

print("EPC shape:", epc_df.shape)
print("Efficiency shape:", eff_df.shape)

EPC shape: (117, 201)
Efficiency shape: (17520, 107)


In [21]:
# STEP 3: Create half-hourly time column
# Description: Generate correct 00:30:00 → 24h loop for full year (17520 rows)

time_steps = pd.date_range(
    start="2026-01-01 00:30:00",
    periods=17520,
    freq="30T"
)

time_strings = time_steps.strftime("%H:%M:%S")

# Create base time DataFrame
time_df = pd.DataFrame({"Time": time_strings})

print(time_df.head())
print("Total timesteps:", len(time_df))

       Time
0  00:30:00
1  01:00:00
2  01:30:00
3  02:00:00
4  02:30:00
Total timesteps: 17520


/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_20307/220444024.py:4: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  time_steps = pd.date_range(


In [22]:
# STEP 4: Validate efficiency table structure
# Description: Ensure required format and alignment

# Check first column is time-like (we won't use it, but good sanity check)
print(eff_df.columns[0])

# Drop first column (time column in efficiency file)
eff_values_df = eff_df.iloc[:, 1:]

# Ensure correct number of rows
assert len(eff_values_df) == 17520, "Efficiency file must have 17520 rows"

print("Available heat codes:", len(eff_values_df.columns))

Time
Available heat codes: 106


In [23]:
# STEP 5: Define helper function to create efficiency CSV
# Description: Extract correct column and save formatted output

def create_efficiency_csv(heat_code, output_path, column_name):
    if heat_code not in eff_values_df.columns:
        print(f"WARNING: Heat code '{heat_code}' not found. Skipping.")
        return
    
    efficiency_series = eff_values_df[heat_code].reset_index(drop=True)
    
    output_df = pd.DataFrame({
        "Time": time_df["Time"],
        column_name: efficiency_series
    })
    
    output_df.to_csv(output_path, index=False)

In [24]:
# STEP 6: Loop through buildings and generate MAIN heating CSVs
# Description: Create main_heat_efficiency.csv for every building

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    main_code = str(row["MAIN_HEAT_CODE"]).strip()
    
    building_path = os.path.join(models_base_path, building_id)
    heating_path = os.path.join(building_path, "HEATING")
    
    # Ensure HEATING folder exists
    if not os.path.exists(heating_path):
        print(f"Missing HEATING folder for {building_id}, skipping.")
        continue
    
    output_file = os.path.join(heating_path, "main_heat_effiency.csv")
    
    create_efficiency_csv(
        heat_code=main_code,
        output_path=output_file,
        column_name="main_heat_efficiency"
    )

print("Main heating files complete.")

Main heating files complete.


In [25]:
# STEP 7: Loop through buildings and generate SECONDARY heating CSVs
# Description: Create second_heat_efficiency.csv only if applicable

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    second_code = str(row["SECOND_HEAT_CODE"]).strip().lower()
    
    # Skip if none
    if second_code == "none" or second_code == "" or pd.isna(second_code):
        continue
    
    building_path = os.path.join(models_base_path, building_id)
    heating_path = os.path.join(building_path, "HEATING")
    
    # Ensure HEATING folder exists
    if not os.path.exists(heating_path):
        print(f"Missing HEATING folder for {building_id}, skipping.")
        continue
    
    output_file = os.path.join(heating_path, "second_heat_effiency.csv")
    
    create_efficiency_csv(
        heat_code=second_code,
        output_path=output_file,
        column_name="second_heat_efficiency"
    )

print("Secondary heating files complete.")

Secondary heating files complete.


In [26]:
# STEP 8: Optional validation check
# Description: Verify one sample output file

sample_building = str(epc_df.iloc[0]["BUILDING_REFERENCE_NUMBER"])
sample_path = os.path.join(models_base_path, sample_building, "HEATING", "main_heat_effiency.csv")

if os.path.exists(sample_path):
    df_check = pd.read_csv(sample_path)
    print(df_check.head())
    print(df_check.shape)
else:
    print("Sample file not found.")

       Time  main_heat_efficiency
0  00:30:00                  0.98
1  01:00:00                  0.98
2  01:30:00                  0.98
3  02:00:00                  0.98
4  02:30:00                  0.98
(17520, 2)
